# 16 Karnataka hospital siting (illustrative)

Question: if ten hospitals were added, which 1 km cells would bring the most people within
10 km (straight line) of a hospital? Plan item 11, task T286.

Inputs: S16 WorldPop 2025 1 km population (CC BY 4.0), S18 OpenStreetMap hospitals from the
owner's Geofabrik extract (ODbL), S17 district boundaries; distances from notebook 11 (D195).
Method: `src/atlas/analysis/siting.py`, greedy maximal coverage over populated cells (within 63%
of the optimum, not guaranteed optimal), against three simple baselines (most populous gap cells,
one site in each worst district, 200 random populated cells), with sensitivity to the number of
sites (5, 10, 20), the threshold (5, 10, 15 km) and dropping 10% of OSM hospitals.

This is illustrative siting under stated assumptions, not a recommendation: straight-line
distance, OSM completeness, no capacity, no land, cost or road data, modelled population.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from atlas.analysis.siting import build

OUT = Path(os.environ.get("ATLAS_EDA_OUT", "reports/eda/16_ka_hospital_siting"))
result = build(OUT)
summary = result["summary"]
print(json.dumps(summary["headline"], indent=1))

In [ ]:
lonlat, km, people = result["lonlat"], result["km"], result["people"]
gap = km > 10
fig, ax = plt.subplots(figsize=(7, 8))
ax.scatter(lonlat[~gap, 0], lonlat[~gap, 1], s=2, c="#cfcfcf", marker="s", linewidths=0)
ax.scatter(
    lonlat[gap, 0],
    lonlat[gap, 1],
    s=2,
    c="#e07b39",
    marker="s",
    linewidths=0,
    label="cell more than 10 km from an OSM hospital",
)
h = result["hospitals"]
ax.scatter(h[:, 0], h[:, 1], s=1, c="#555555", linewidths=0, label="OSM hospital")
c = result["chosen"]
ax.scatter(c[:, 0], c[:, 1], s=60, marker="*", c="#1d3557", label="greedy pick (illustrative)")
for i, (x, y) in enumerate(c, 1):
    ax.annotate(str(i), (x, y), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_aspect("equal")
ax.set_title("Karnataka: ten illustrative hospital cells by greedy 10 km coverage")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.legend(loc="lower left", fontsize=7)
fig.text(
    0.01,
    0.01,
    "S16 WorldPop 2025 (CC BY 4.0), S18 OpenStreetMap (ODbL). Straight-line; OSM coverage "
    "varies; not a recommendation.",
    fontsize=6,
)
fig.savefig(OUT / "siting_map.png", dpi=80, metadata={"Software": None})
plt.close(fig)

In [ ]:
curve = result["curve"]
labels = {
    "greedy": "greedy maximal coverage",
    "most_populous_gap_cells": "most populous gap cells",
    "one_per_worst_district": "one per worst district",
    "random_mean": "random populated cells (mean of 200)",
}
fig, ax = plt.subplots(figsize=(6, 4.5))
for name, label in labels.items():
    y = np.asarray(curve[name]) / 1e6
    ax.plot(np.arange(1, len(y) + 1), y, marker="o", markersize=3, label=label)
ax.set_xlabel("new hospital sites")
ax.set_ylabel("people newly within 10 km (millions)")
ax.set_title("People brought within 10 km, by number of sites")
ax.set_xticks([1, 5, 10, 15, 20])
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "gain_curve.png", dpi=80, metadata={"Software": None})
plt.close(fig)
for row in summary["sensitivity"]:
    print(row)

## Limits

See `summary.json` fields `limits`, `baseline_not_run` and `osm_completeness_check`. Greedy
coverage beats every baseline here, but dropping 10% of OSM hospitals moves several of the ten
picks, so read the cells as "areas where a hospital would close the largest straight-line gap",
not as sites. Not an emergency service and no claim that any location is safe or adequate.